In [ ]:

import pandas as pd
import numpy as np
from lifetimes.utils import summary_data_from_transaction_data
from lifetimes import BetaGeoFitter
from lifetimes import GammaGammaFitter

# Load data
df = pd.read_csv("lifetimes_summary_with_CLV.csv")

# Convert data to a dataframe and name it "lifetimes_summary"
lifetimes_summary = df

# Segments per CLV

In [2]:
lifetimes_summary['CLV_6m'].describe()

count    8.755580e+05
mean     1.465119e+01
std      2.791636e+01
min      4.962536e-73
25%      3.366665e+00
50%      5.679045e+00
75%      1.415012e+01
max      2.803953e+03
Name: CLV_6m, dtype: float64

In [3]:
repeaters = lifetimes_summary[lifetimes_summary["frequency"] > 0]
clv_75_repeaters = repeaters["CLV_6m"].quantile(0.75)

def assign_segment(row):
    if row["frequency"] == 0:
        return "One-timer"
    elif row["CLV_6m"] < clv_75_repeaters:
        return "Casual"
    else:
        return "Star"

lifetimes_summary["segment"] = lifetimes_summary.apply(assign_segment, axis=1)

In [4]:
# see the count of each segment
lifetimes_summary["segment"].value_counts()

segment
One-timer    549274
Casual       244713
Star          81571
Name: count, dtype: int64

In [5]:
# see the percentage of each segment
lifetimes_summary["segment"].value_counts(normalize=True) * 100

segment
One-timer    62.734165
Casual       27.949376
Star          9.316459
Name: proportion, dtype: float64

# Distribution of Segments, per Channel

how many each customers are in which segments? segments = row, channels = columns

In [6]:
lifetimes_summary = lifetimes_summary.reset_index()

customer_info = df.groupby('CustomerID').agg({
    'Channel': lambda x: x.mode()[0],
    'StoreName': lambda x: x.mode()[0]
}).reset_index()

lifetimes_summary = lifetimes_summary.merge(
    customer_info,
    on='CustomerID',
    how='left'
)

# If duplciated columns:
lifetimes_summary = lifetimes_summary.rename(columns={
    'Channel_x': 'Channel',
    'StoreName_x': 'StoreName'
})

lifetimes_summary = lifetimes_summary.drop(columns=['Channel_y', 'StoreName_y'], errors='ignore')

In [7]:
# Distrubution of segments across channels
channel_segment = pd.crosstab(lifetimes_summary['Channel'], lifetimes_summary['segment'])
channel_segment_pct = pd.crosstab(lifetimes_summary['Channel'], lifetimes_summary['segment'], normalize='index') * 100

channel_segment_summary = pd.concat([
    channel_segment,
    channel_segment_pct.add_suffix('_%')
], axis=1).sort_index(axis=1)

print(channel_segment_summary)

segment              Casual   Casual_%  One-timer  One-timer_%   Star  \
Channel                                                                 
ConsumerApp            5509  59.274801       3559    38.293523    226   
Intake module          3338  20.121767      13217    79.673277     34   
S4D - Cash register       0   0.000000          6   100.000000      0   
TakeAway             174804  28.695438     376630    61.826748  57736   
UberEats               3693  21.705654      13236    77.794757     85   
Website               57369  25.670179     142626    63.819048  23490   

segment                 Star_%  
Channel                         
ConsumerApp           2.431676  
Intake module         0.204955  
S4D - Cash register   0.000000  
TakeAway              9.477814  
UberEats              0.499589  
Website              10.510773  


In [8]:
# Distribution of channels across segments
segment_channel = pd.crosstab(lifetimes_summary['segment'], lifetimes_summary['Channel'])
segment_channel_pct = pd.crosstab(lifetimes_summary['segment'], lifetimes_summary['Channel'], normalize='index') * 100

segment_channel_summary = pd.concat([
    segment_channel,
    segment_channel_pct.add_suffix('_%')
], axis=1).sort_index(axis=1)

print(segment_channel_summary)

Channel    ConsumerApp  ConsumerApp_%  Intake module  Intake module_%  \
segment                                                                 
Casual            5509       2.251209           3338         1.364047   
One-timer         3559       0.647946          13217         2.406267   
Star               226       0.277059             34         0.041681   

Channel    S4D - Cash register  S4D - Cash register_%  TakeAway  TakeAway_%  \
segment                                                                       
Casual                       0               0.000000    174804   71.432249   
One-timer                    6               0.001092    376630   68.568692   
Star                         0               0.000000     57736   70.780057   

Channel    UberEats  UberEats_%  Website  Website_%  
segment                                              
Casual         3693    1.509115    57369  23.443381  
One-timer     13236    2.409726   142626  25.966275  
Star             85   

# Distribution of Segments, per Geographical Store

In [9]:
# Distribution of stores across segments
store_segment = pd.crosstab(lifetimes_summary['StoreName'], lifetimes_summary['segment'])
store_segment_pct = pd.crosstab(lifetimes_summary['StoreName'], lifetimes_summary['segment'], normalize='index') * 100

store_segment_summary = pd.concat([
    store_segment,
    store_segment_pct.add_suffix('_%')
], axis=1).sort_index(axis=1)

# I add a total column and just show the top 10 stores by volume
store_segment_summary['Total_Customers'] = store_segment.sum(axis=1)
print(store_segment_summary.sort_values('Total_Customers', ascending=False).head(10))

segment                           Casual   Casual_%  One-timer  One-timer_%  \
StoreName                                                                     
Spare Rib Express Tilburg           6611  28.324764      14519    62.206512   
Spare Rib Express Alkmaar           6753  29.013963      14495    62.277121   
Spare Rib Express Purmerend         7299  33.707398      12312    56.857855   
Spare Rib Express Rotterdam         5264  26.482870      13154    66.176988   
Spare Rib Express Den Bosch         5374  28.302086      11970    63.039815   
Spare Rib Express Breda             5046  28.038006      11287    62.716008   
Spare Rib Express Middelburg        5160  29.708101       9932    57.182336   
Spare Rib Express Zoetermeer        5072  29.445573      10828    62.862119   
Spare Rib Express Bergen Op Zoom    5213  30.751534       9922    58.529967   
Spare Rib Express Leeuwarden        4717  28.042328      10270    61.054634   

segment                           Star     Star_%  

In [10]:
# Distribution of stores across segments
# normalize='columns' means the percentages will add up to 100% at the bottom of the columns
store_segment_col_pct = pd.crosstab(lifetimes_summary['StoreName'], lifetimes_summary['segment'], normalize='columns') * 100

# See where each segments are heavily concentrated by sheer volume
star_locations = store_segment_col_pct[['Star']].sort_values(by='Star', ascending=False)
print("Percentage of the total Star segment located at each store:")
print(star_locations.head(10))

Percentage of the total Star segment located at each store:
segment                               Star
StoreName                                 
Spare Rib Express Middelburg      2.791433
Spare Rib Express Tilburg         2.709296
Spare Rib Express Purmerend       2.504567
Spare Rib Express Alkmaar         2.484952
Spare Rib Express Leeuwarden      2.248348
Spare Rib Express Bergen Op Zoom  2.227507
Spare Rib Express Hoorn           2.146596
Spare Rib Express Den Helder      2.092655
Spare Rib Express Breda           2.039941
Spare Rib Express Den Bosch       2.015422


In [11]:
casuals_locations = store_segment_col_pct[['Casual']].sort_values(by='Casual', ascending=False)
print("Percentage of the total Casual segment located at each store:")
print(casuals_locations.head(10))

Percentage of the total Casual segment located at each store:
segment                             Casual
StoreName                                 
Spare Rib Express Purmerend       2.982678
Spare Rib Express Alkmaar         2.759559
Spare Rib Express Tilburg         2.701532
Spare Rib Express Den Bosch       2.196042
Spare Rib Express Rotterdam       2.151091
Spare Rib Express Bergen Op Zoom  2.130251
Spare Rib Express Middelburg      2.108593
Spare Rib Express Zoetermeer      2.072632
Spare Rib Express Breda           2.062007
Spare Rib Express Leeuwarden      1.927564


In [12]:
onetimer_locations = store_segment_col_pct[['One-timer']].sort_values(by='One-timer', ascending=False)
print("Percentage of the total One-timer segment located at each store:")
print(onetimer_locations.head(10))

Percentage of the total One-timer segment located at each store:
segment                       One-timer
StoreName                              
Spare Rib Express Tilburg      2.643307
Spare Rib Express Alkmaar      2.638938
Spare Rib Express Rotterdam    2.394797
Spare Rib Express Purmerend    2.241504
Spare Rib Express Den Bosch    2.179240
Spare Rib Express Breda        2.054894
Spare Rib Express Zoetermeer   1.971329
Spare Rib Express Utrecht      1.914345
Spare Rib Express Leeuwarden   1.869741
Spare Rib Express Hoorn        1.831326


# Acquisition assessments


## Channel quality
Which acquisition channels yield the highest percentage of "Stars"?

In [13]:
# Calculate the percentage of Stars generated by each Channel
channel_quality = channel_segment_pct[['Star']].sort_values(by='Star', ascending=False)
print("Percentage of customers who become Stars, per channel:")
print(channel_quality)

Percentage of customers who become Stars, per channel:
segment                   Star
Channel                       
Website              10.510773
TakeAway              9.477814
ConsumerApp           2.431676
UberEats              0.499589
Intake module         0.204955
S4D - Cash register   0.000000


## Average predicted value per Channel
When you acquire a new customer through a specific channel, how much are they worth over the next 6 months?

In [14]:
# Average 6m CLV by Channel
channel_value = lifetimes_summary.groupby('Channel')['CLV_6m'].mean().sort_values(ascending=False)
print("Average 6m CLV per Channel:")
print(channel_value)

Average 6m CLV per Channel:
Channel
Website                15.793697
TakeAway               14.932903
ConsumerApp             7.186103
UberEats                4.305305
Intake module           3.712196
S4D - Cash register     2.422052
Name: CLV_6m, dtype: float64


## Store-level acquisition efficiency
A hypothesis is that some stores might be naturally better at acquiring high-value customers due to their local demographics. Identifying the top 5 "star-producing" stores can help in targeting local marketing efforts like flyers or local social media ads.

In [15]:
# Look at stores with the highest concentration of Stars (min 100 customers to avoid outliers)
valid_stores = store_segment_summary[store_segment_summary['Total_Customers'] > 100]
top_star_stores = valid_stores.sort_values(by='Star_%', ascending=False).head(10)
print("Top 10 Stores by percentage of Star:")
print(top_star_stores[['Star_%', 'Total_Customers']])

Top 10 Stores by percentage of Star:
segment                            Star_%  Total_Customers
StoreName                                                 
Spare Rib Express Heerlen       22.324511              869
Spare Rib Express Middelburg    13.109563            17369
Spare Rib Express Leidschendam  12.972572             2698
Spare Rib Express Geldermalsen  12.737334            10481
Spare Rib Express Den Helder    12.519252            13635
Spare Rib Express Terneuzen     11.716477            11343
Spare Rib Express Kampen        11.677019             8855
Spare Rib Express Beverwijk     11.350652             9277
Spare Rib Express Zeist         11.247978             8037
Spare Rib Express Hoorn         11.102657            15771
